# Notebook 11 — Reviewer R1.4 Figure 3 Contrast Revision

**Reviewer comment R1.4**

> “The NTL visualization in Figure 3 should use a more appropriate contrast stretch, as the current display makes spatial variations in NTL radiance difficult to distinguish.”

This notebook regenerates Figure 3 as four separate two-panel exports for 1, 5, 10, and 13 November 2013. Each export contains `DNB_BRDF_Corrected_NTL` and `Mandatory_Quality_Flag`. A shared display range is applied across all dates; statistics are calculated from the untransformed radiance arrays. Individual files are assembled and subcaptioned in Overleaf.

## 1. Imports

In [1]:
from pathlib import Path
import gc
import os

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

## 2. Project paths and output folders

In [2]:
DEFAULT_ROOT = Path(
    "/Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/"
    "02 - CH1 - Reliability-Qualified Nighttime Lights (Technical Validation)/"
    "blackmarble-reliability-conditions"
)

cwd_root = Path.cwd()
if cwd_root.name == "notebooks":
    cwd_root = cwd_root.parent

PROJECT_ROOT = cwd_root if (cwd_root / "datasets").exists() else Path(
    os.environ.get("RQ1_PROJECT_ROOT", DEFAULT_ROOT)
).expanduser()

DATASET_DIR = PROJECT_ROOT / "datasets"
DAILY_TIFF_DIR = DATASET_DIR / "vnp46a2" / "region8_allbands_nov2013"
REGIONS_PATH = DATASET_DIR / "boundaries" / "regions" / "Regions.shp"

OUTPUT_MAIN_FIG_DIR = PROJECT_ROOT / "outputs" / "figures" / "main"
OUTPUT_SUPP_TABLE_DIR = PROJECT_ROOT / "outputs" / "tables" / "supplementary"

for output_dir in [OUTPUT_MAIN_FIG_DIR, OUTPUT_SUPP_TABLE_DIR]:
    output_dir.mkdir(parents=True, exist_ok=True)

required_paths = [DAILY_TIFF_DIR, REGIONS_PATH]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing required inputs: {missing_paths}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Daily GeoTIFFs: {DAILY_TIFF_DIR}")
print(f"Regional boundaries: {REGIONS_PATH}")
print(f"Figure output: {OUTPUT_MAIN_FIG_DIR}")

Project root: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH1 - Reliability-Qualified Nighttime Lights (Technical Validation)/blackmarble-reliability-conditions
Daily GeoTIFFs: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH1 - Reliability-Qualified Nighttime Lights (Technical Validation)/blackmarble-reliability-conditions/datasets/vnp46a2/region8_allbands_nov2013
Regional boundaries: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH1 - Reliability-Qualified Nighttime Lights (Technical Validation)/blackmarble-reliability-conditions/datasets/boundaries/regions/Regions.shp
Figure output: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH1 - Reliability-Qualified Nighttime Lights (Technical Validation)/blackmarble-reliability-conditions/outputs/figures/main


## 3. Figure configuration

In [3]:
TARGET_DATES = {
    "20131101": {"panel": "a", "label": "1 Nov 2013"},
    "20131105": {"panel": "b", "label": "5 Nov 2013"},
    "20131110": {"panel": "c", "label": "10 Nov 2013"},
    "20131113": {"panel": "d", "label": "13 Nov 2013"},
}

# Rasterio uses one-based band indexing.
BAND_DNB_BRDF = 1
BAND_GAP_FILLED = 3
BAND_MQF = 5

DISPLAY_PERCENTILE = 99
FIGURE_WIDTH = 820
FIGURE_HEIGHT = 500
EXPORT_SCALE = 2

QF_LABELS = [
    "Persistent (0)",
    "Ephemeral (1)",
    "Poor/Cloud (2)",
    "Fill (255)",
]

QF_COLORSCALE = [
    [0.0000, "green"], [0.2499, "green"],
    [0.2500, "royalblue"], [0.4999, "royalblue"],
    [0.5000, "orange"], [0.7499, "orange"],
    [0.7500, "lightgrey"], [1.0000, "lightgrey"],
]

## 4. Lightweight helper functions

In [4]:
def find_daily_tiff(date_key):
    matches = sorted(DAILY_TIFF_DIR.glob(f"*{date_key}*.tif"))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one GeoTIFF for {date_key}; found {len(matches)}: {matches}"
        )
    return matches[0]


def read_band(src, band_number):
    return src.read(band_number, masked=True).filled(np.nan).astype(np.float32)


def export_plotly_figure(fig, output_path, width, height, scale=2):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    fig.write_image(
        str(output_path.with_suffix(".png")),
        width=width,
        height=height,
        scale=scale,
    )
    fig.write_image(
        str(output_path.with_suffix(".svg")),
        width=width,
        height=height,
        scale=1,
    )

    print(f"Exported: {output_path.with_suffix('.png')}")
    print(f"Exported: {output_path.with_suffix('.svg')}")


def append_boundary_coordinates(geometry, x_values, y_values):
    if geometry is None or geometry.is_empty:
        return

    if geometry.geom_type == "LineString":
        x_coords, y_coords = geometry.xy
        x_values.extend(x_coords)
        y_values.extend(y_coords)
        x_values.append(None)
        y_values.append(None)
        return

    if geometry.geom_type in {"Polygon", "MultiPolygon"}:
        append_boundary_coordinates(geometry.boundary, x_values, y_values)
        return

    if hasattr(geometry, "geoms"):
        for part in geometry.geoms:
            append_boundary_coordinates(part, x_values, y_values)

## 5. Input inventory

In [5]:
TIFF_BY_DATE = {
    date_key: find_daily_tiff(date_key)
    for date_key in TARGET_DATES
}

inventory_rows = []
for date_key, path in TIFF_BY_DATE.items():
    with rasterio.open(path) as src:
        inventory_rows.append({
            "date": pd.to_datetime(date_key, format="%Y%m%d"),
            "file": path.name,
            "bands": src.count,
            "height": src.height,
            "width": src.width,
            "crs": str(src.crs),
        })

INPUT_INVENTORY = pd.DataFrame(inventory_rows)
display(INPUT_INVENTORY)

,date,file,bands,height,width,crs
0,2013-11-01,VNP46A2_Region8_20131101.tif,7,639,439,EPSG:4326
1,2013-11-05,VNP46A2_Region8_20131105.tif,7,639,439,EPSG:4326
2,2013-11-10,VNP46A2_Region8_20131110.tif,7,639,439,EPSG:4326
3,2013-11-13,VNP46A2_Region8_20131113.tif,7,639,439,EPSG:4326


## 6. Shared map extent and regional boundary

In [6]:
with rasterio.open(TIFF_BY_DATE["20131101"]) as src:
    left, bottom, right, top = src.bounds

MAP_LON_RANGE = [float(left), float(right)]
MAP_LAT_RANGE = [float(bottom), float(top)]

regions_gdf = gpd.read_file(REGIONS_PATH).to_crs(epsg=4326)
visible_regions = regions_gdf.cx[left:right, bottom:top]

if visible_regions.empty:
    raise ValueError("Regions.shp does not intersect the daily raster extent")

BOUNDARY_X = []
BOUNDARY_Y = []

for geometry in visible_regions.geometry:
    append_boundary_coordinates(geometry, BOUNDARY_X, BOUNDARY_Y)

del regions_gdf, visible_regions
gc.collect()

print(f"Boundary coordinate count: {len(BOUNDARY_X):,}")

Boundary coordinate count: 248,210


## 7. Shared display range

The NTL panels use a common linear range from 0 to the pooled 99th percentile across the four displayed dates. This is a display-only stretch. Mean and median statistics are calculated from the original DNB-BRDF arrays.

In [7]:
contrast_values = []

for path in TIFF_BY_DATE.values():
    with rasterio.open(path) as src:
        ntl = read_band(src, BAND_DNB_BRDF)
        gap = read_band(src, BAND_GAP_FILLED)

    valid = np.isfinite(ntl) & np.isfinite(gap)
    contrast_values.append(ntl[valid])
    del ntl, gap, valid

contrast_values = np.concatenate(contrast_values)

DISPLAY_MIN = 0.0
DISPLAY_MAX = float(np.nanpercentile(contrast_values, DISPLAY_PERCENTILE))
if not np.isfinite(DISPLAY_MAX) or DISPLAY_MAX <= DISPLAY_MIN:
    DISPLAY_MAX = 1.0

NTL_TICKS = np.linspace(DISPLAY_MIN, DISPLAY_MAX, 6)

del contrast_values
gc.collect()

print(f"Shared display range: {DISPLAY_MIN:.2f}–{DISPLAY_MAX:.2f} nW/cm²/sr")

Shared display range: 0.00–1.01 nW/cm²/sr


## 8. Figure 3 builder

In [8]:
def build_daily_figure(date_key):
    path = TIFF_BY_DATE[date_key]

    with rasterio.open(path) as src:
        ntl = np.flipud(read_band(src, BAND_DNB_BRDF))
        gap = np.flipud(read_band(src, BAND_GAP_FILLED))
        mqf = np.flipud(read_band(src, BAND_MQF))

        left, bottom, right, top = src.bounds
        lons = np.linspace(left, right, src.width, dtype=np.float32)
        lats = np.linspace(bottom, top, src.height, dtype=np.float32)

    land_mask = np.isfinite(gap)
    valid_ntl_mask = land_mask & np.isfinite(ntl)

    total_land_px = int(land_mask.sum())
    valid_ntl_px = int(valid_ntl_mask.sum())
    spatial_completeness = (
        100 * valid_ntl_px / total_land_px if total_land_px else np.nan
    )

    mean_ntl = float(np.nanmean(ntl[valid_ntl_mask]))
    median_ntl = float(np.nanmedian(ntl[valid_ntl_mask]))
    mean_gap = float(np.nanmean(gap[land_mask]))
    median_gap = float(np.nanmedian(gap[land_mask]))

    mqf_plot = np.full(mqf.shape, np.nan, dtype=np.float32)
    mqf_land = mqf[land_mask].copy()
    mqf_land[~np.isfinite(mqf_land)] = 255
    mqf_land[mqf_land == 255] = 3
    mqf_plot[land_mask] = mqf_land
    del mqf_land

    fig = make_subplots(
        rows=1,
        cols=2,
        shared_yaxes=True,
        horizontal_spacing=0.15,
    )

    fig.add_trace(
        go.Heatmap(
            z=ntl,
            x=lons,
            y=lats,
            zmin=DISPLAY_MIN,
            zmax=DISPLAY_MAX,
            colorscale="Cividis",
            colorbar=dict(
                title=dict(text="nW/cm²/sr", side="top", font=dict(size=12)),
                tickvals=NTL_TICKS,
                ticktext=[f"{value:.1f}" for value in NTL_TICKS],
                tickfont=dict(size=10),
                len=0.50,
                thickness=13,
                x=0.475,
                y=0.66,
            ),
            hoverinfo="skip",
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Heatmap(
            z=mqf_plot,
            x=lons,
            y=lats,
            zmin=-0.5,
            zmax=3.5,
            colorscale=QF_COLORSCALE,
            colorbar=dict(
                title=dict(text="Quality Flag", side="top", font=dict(size=12)),
                tickvals=[0, 1, 2, 3],
                ticktext=QF_LABELS,
                tickfont=dict(size=10),
                len=0.50,
                thickness=13,
                x=1.025,
                y=0.66,
            ),
            hoverinfo="skip",
        ),
        row=1,
        col=2,
    )

    for column in [1, 2]:
        fig.add_trace(
            go.Scatter(
                x=BOUNDARY_X,
                y=BOUNDARY_Y,
                mode="lines",
                line=dict(color="rgba(35,35,35,0.90)", width=1.1),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=column,
        )

    fig.add_annotation(
        text="DNB_BRDF_Corrected_NTL",
        xref="x domain",
        yref="y domain",
        x=0,
        y=1.035,
        xanchor="left",
        yanchor="bottom",
        showarrow=False,
        font=dict(family="Arial", size=13, color="black"),
    )
    fig.add_annotation(
        text="Mandatory_Quality_Flag",
        xref="x2 domain",
        yref="y2 domain",
        x=0,
        y=1.035,
        xanchor="left",
        yanchor="bottom",
        showarrow=False,
        font=dict(family="Arial", size=13, color="black"),
    )

    fig.add_annotation(
        text=(
            f"DNB-BRDF — Mean: {mean_ntl:.2f}; Median: {median_ntl:.2f}<br>"
            f"Gap-filled — Mean: {mean_gap:.2f}; Median: {median_gap:.2f}"
        ),
        xref="x domain",
        yref="paper",
        x=0.5,
        y=-0.14,
        xanchor="center",
        yanchor="top",
        showarrow=False,
        align="center",
        font=dict(family="Arial", size=10, color="black"),
    )
    fig.add_annotation(
        text=f"Spatial Completeness: {spatial_completeness:.1f}%",
        xref="x2 domain",
        yref="paper",
        x=0.5,
        y=-0.14,
        xanchor="center",
        yanchor="top",
        showarrow=False,
        font=dict(family="Arial", size=10, color="black"),
    )

    fig.update_layout(
        template="plotly_white",
        width=FIGURE_WIDTH,
        height=FIGURE_HEIGHT,
        margin=dict(l=50, r=175, t=55, b=105),
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="#E5ECF6",
        font=dict(family="Arial", size=11, color="black"),
    )
    fig.update_xaxes(
        range=MAP_LON_RANGE,
        tickformat=".1f",
        tickfont=dict(size=9),
        showgrid=True,
        gridcolor="rgba(255,255,255,0.70)",
        zeroline=False,
    )
    fig.update_yaxes(
        range=MAP_LAT_RANGE,
        tickformat=".1f",
        tickfont=dict(size=9),
        showgrid=True,
        gridcolor="rgba(255,255,255,0.70)",
        zeroline=False,
    )
    fig.update_yaxes(showticklabels=False, row=1, col=2)

    statistics = {
        "date": pd.to_datetime(date_key, format="%Y%m%d"),
        "valid_pixels": valid_ntl_px,
        "total_land_pixels": total_land_px,
        "spatial_completeness_pct": spatial_completeness,
        "dnb_brdf_mean": mean_ntl,
        "dnb_brdf_median": median_ntl,
        "gap_filled_mean": mean_gap,
        "gap_filled_median": median_gap,
    }

    del ntl, gap, mqf, mqf_plot, land_mask, valid_ntl_mask
    return fig, statistics

## 9. Export the four individual panels

Figures are created and released sequentially. They are not retained in a list and are not displayed inline, preventing the notebook from storing four large Plotly JSON outputs in memory.

In [9]:
result_rows = []

for date_key, metadata in TARGET_DATES.items():
    fig, statistics = build_daily_figure(date_key)

    output_stem = (
        OUTPUT_MAIN_FIG_DIR
        / f"fig_03{metadata['panel']}_vnp46a2_daily_{date_key}"
    )

    export_plotly_figure(
        fig,
        output_stem,
        width=FIGURE_WIDTH,
        height=FIGURE_HEIGHT,
        scale=EXPORT_SCALE,
    )

    statistics["panel"] = metadata["panel"]
    statistics["label"] = metadata["label"]
    statistics["output_stem"] = str(output_stem)
    result_rows.append(statistics)

    del fig
    gc.collect()

FIGURE_3_EVIDENCE = pd.DataFrame(result_rows).sort_values("date")
evidence_path = OUTPUT_SUPP_TABLE_DIR / "figure_03_daily_panel_evidence.csv"
FIGURE_3_EVIDENCE.to_csv(evidence_path, index=False)

display(FIGURE_3_EVIDENCE)
print(f"Exported evidence table: {evidence_path}")

Exported: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH1 - Reliability-Qualified Nighttime Lights (Technical Validation)/blackmarble-reliability-conditions/outputs/figures/main/fig_03a_vnp46a2_daily_20131101.png
Exported: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH1 - Reliability-Qualified Nighttime Lights (Technical Validation)/blackmarble-reliability-conditions/outputs/figures/main/fig_03a_vnp46a2_daily_20131101.svg
Exported: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH1 - Reliability-Qualified Nighttime Lights (Technical Validation)/blackmarble-reliability-conditions/outputs/figures/main/fig_03b_vnp46a2_daily_20131105.png
Exported: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH1 - Reliability-Qualified Nighttime Lights (Technical Validation)/blackmarble-reliability-conditions/outputs/figures/main/fig_03b_vnp46a2_daily_20131105.svg
Exported: /Users/reneprincipejr/Libr

,date,valid_pixels,total_land_pixels,spatial_completeness_pct,dnb_brdf_mean,dnb_brdf_median,gap_filled_mean,gap_filled_median,panel,label,output_stem
0,2013-11-01,61107,89614,68.189122,0.324104,0.226140,0.282541,0.173741,a,1 Nov 2013,/Users/reneprincipejr/Library/CloudStorage/One...
1,2013-11-05,21584,89614,24.085522,0.169343,0.125175,0.187224,0.123056,b,5 Nov 2013,/Users/reneprincipejr/Library/CloudStorage/One...
2,2013-11-10,70151,89614,78.281295,0.134760,0.127802,0.140328,0.123921,c,10 Nov 2013,/Users/reneprincipejr/Library/CloudStorage/One...
3,2013-11-13,87104,89614,97.199098,0.058920,0.010000,0.061805,0.010000,d,13 Nov 2013,/Users/reneprincipejr/Library/CloudStorage/One...


Exported evidence table: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH1 - Reliability-Qualified Nighttime Lights (Technical Validation)/blackmarble-reliability-conditions/outputs/tables/supplementary/figure_03_daily_panel_evidence.csv


## 10. Optional single-panel preview

In [11]:
# Preview only one date when needed. Keeping this disabled avoids large
# inline notebook outputs during routine export.
PREVIEW_DATE = None  # Example: "20131101"

if PREVIEW_DATE is not None:
    if PREVIEW_DATE not in TARGET_DATES:
        raise ValueError(f"Choose one of: {list(TARGET_DATES)}")

    preview_fig, _ = build_daily_figure(PREVIEW_DATE)
    preview_fig.show()
    del preview_fig
    gc.collect()